<a href="https://colab.research.google.com/github/durga0000/RAG/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

if 'COLAB_GPU' in os.environ:
  print('[info] Running in google colab , installing requirements.')
  !pip install pyMuPDF
  !pip install tqdm
  !pip install accelerate
  !pip install bitsandbytes
  !pip install flash-attn --no-build-isolation

In [ ]:
!pip uninstall -y torch torchvision torchaudio transformers sentence-transformers

!pip install torch torchvision torchaudio \
    --index-url https://download.pytorch.org/whl/cu121

!pip install -U transformers sentence-transformers

In [ ]:
import os

In [ ]:
import requests

pdf_path = 'human nutrition.pdf'

if not os.path.exists(pdf_path):
   print('File does not exist , downloaging...')

   url = "https://www.cur.ac.rw/mis/main/library/documents/book_file/digital-67fe1efc6feac4.39858496.pdf"

   filename = pdf_path

   response = requests.get(url)

   if response.status_code == 200:
       with open(filename , "wb") as file:
             file.write(response.content)
       print(f"file has been downloaded and saved as {filename}")
   else:
       print(f"failed to download the file . staus code: {response.status_code}")

else:
  print(f"file {pdf_path} exists.")


In [ ]:
import fitz
from tqdm.auto import tqdm

def text_formatter(text: str)->str:
    cleaned_text = text.replace("\n"," ").strip()

    return cleaned_text

def open_and_read_pdf(pdf_path:str)->list[dict]:

     doc  = fitz.open(pdf_path)
     pages_and_texts = []

     for page_number , page in tqdm(enumerate(doc)):
         text = page.get_text()
         text = text_formatter(text)

         pages_and_texts.append({"page_number":page_number-8,
                                 "physical_page": page_number,
                                 "page_char_count":len(text),
                                 "page_word_count":len(text.split(' ')),
                                 "page_sentence_count_raw":len(text.split(".  ")),
                                 "page_token_count":len(text)/4 ,
                                 "text": text
                                 })
     return pages_and_texts

pages_and_texts = open_and_read_pdf(pdf_path=pdf_path)

pages_and_texts[:10]


In [ ]:
import pandas as pd

df = pd.DataFrame(pages_and_texts)
df.head()

In [ ]:
df.describe().round(2)